# Data Analysis and Visualisation

## Lab 5: Clean the bike rentals

### Preliminaries

In [ ]:
import pandas as pd

### Part 1

#### Task 1: Load it, and remove the empty columns

In [ ]:
rentals_raw = pd.read_csv("data/bike_rentals.csv")
rentals = rentals_raw.copy()

print(rentals.shape)
print(rentals.dtypes)

print("Empty columns:", rentals.columns[rentals.isna().all()])

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object
Empty columns: Index(['coupon_code', 'damage_notes', 'late_fee_note'], dtype='str')


Column "price" is text (`str`). Columns "coupon_code", "damage_notes" and "late_fee_note" are completely empty.

In [ ]:
rentals = rentals.dropna(axis=1, how="all")
print(rentals.shape)
print("How many empty columns are left:", rentals.isna().all().sum())

(403, 12)
How many empty columns are left: 0


After the tree empty columns are removed, there are 12 columns left.

#### Task 2: The price: from text to numbers

In [ ]:
rentals["price"].head(10)

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
3      SEK 569.50
4      SEK 828.00
5      SEK 246.00
6      SEK 476.00
7    SEK 1,838.50
8    SEK 1,555.00
9      SEK 189.50
Name: price, dtype: str

In [ ]:
rentals["price"] = rentals["price"].str.replace("SEK ", "", regex=False)
print(rentals["price"].head(3))

0      855.50
1    1,311.50
2      350.00
Name: price, dtype: str


In [ ]:
rentals["price"] = rentals["price"].str.replace(",", "", regex=False)
print(rentals["price"].head(3))

0     855.50
1    1311.50
2     350.00
Name: price, dtype: str


In [ ]:
rentals["price"] = rentals["price"].astype(float)
print(rentals["price"].dtype)
print(rentals["price"].describe())

float64
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64


In [ ]:
print(rentals.nlargest(3, "price")[["rental_id", "bike_type", "status", "price"]].round(2))

    rental_id   bike_type     status   price
187    R-5282  Cargo bike  Completed  3401.0
49     R-5209  Cargo bike  Completed  2979.5
270    R-5378      E-bike  Completed  2941.0


#### Task 3: The missing prices

In [ ]:
print(rentals["price"].isna().sum())
print((rentals["price"].isna().mean() * 100).round(1))
pd.crosstab(rentals["price"].isna(), rentals["status"] == "Cancelled")

51
12.7


status,False,True
price,,
False,352,0
True,4,47


There are 51 prices that are missing (12.7 %). 47 of them are cancelled rentals, missing for a reason: nothing was paid. 4 are not cancelled, so they are missing by accident.

We leave all of them as missing and do not fill them. A made-up price would change the mean and median, and the median is then based on 352 rentals, not 403.

#### Task 4: The letters "t" and "f"

In [ ]:
tf_columns = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {"t", "f"}]
print(tf_columns)

print(rentals["status"].unique())

['is_member', 'insured', 'paid_online']
<StringArray>
['Completed', 'No-show', 'Cancelled']
Length: 3, dtype: str


Column "status" is not in the list because it has three different texts (Completed, No-show, Cancelled).

In [ ]:
for c in tf_columns:
    rentals[c] = rentals[c].map({"t": True, "f": False})

print(rentals[tf_columns].isna().sum())

is_member      0
insured        3
paid_online    0
dtype: int64


After `map` only "insured" has missing values (3).

In [ ]:
display(rentals.groupby("is_member")["price"].agg(n="count", median="median"))

,n,median
is_member,,
False,248,756.25
True,104,558.00


Members pay less (median 558, 104 rentals) than non-members (756, 248 rentals), but this does not show that membership is the reason.

#### Task 5: The dates

In [ ]:
print("Missing values in the dates columns before:")
print(rentals[["start_date", "end_date"]].isna().sum())

Missing values in the dates columns before:
start_date    0
end_date      0
dtype: int64


In [ ]:
rentals["start_date"] = pd.to_datetime(rentals["start_date"], errors="coerce")
rentals["end_date"] = pd.to_datetime(rentals["end_date"], errors="coerce")

print("Missing values in the dates columns after:")
print(rentals[["start_date", "end_date"]].isna().sum())
print(rentals[["start_date", "end_date"]].dtypes)

Missing values in the dates columns after:
start_date    2
end_date      0
dtype: int64
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object


Before there were no missing dates; after, "start_date" has 2 missing: those were impossible dates (e.g. 30 February) that `errors="coerce"` turned into `NaT`.

In [ ]:
rentals["rental_days"] = (rentals["end_date"] - rentals["start_date"]).dt.days
print(rentals["rental_days"].describe())

count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


The minimum of "rental_days" is -2: some rentals end before they start, which cannot be true.

#### Task 6: The missing ratings

In [ ]:
# How many missing ratings are there
print("Missing ratings:", rentals["rating"].isna().sum())
# Compare missing ratings with status
display(pd.crosstab(rentals["rating"].isna(), rentals["status"]))

Missing ratings: 115


status,Cancelled,Completed,No-show
rating,,,
False,0,288,0
True,47,46,22


There are 115 missing ratings: 47 cancelled, 22 no-shows and 46 completed. It is not a perfect match: cancelled rentals and no-shows have no rating for a reason, but a completed rental can be without one because the customer did not give a rating. So the 46 completed rentals without a rating are the ones where the missing value is real information about customer behaviour, not a data problem.

In [ ]:
# Create a column boolean "has_rating", and count it
rentals["has_rating"] = rentals["rating"].notna()
print(rentals["has_rating"].value_counts())
# Describe the values in the "ratings" column
display(rentals["rating"].describe())

has_rating
True     288
False    115
Name: count, dtype: int64


count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

The median rating is 5.0.

#### Task 7: The helmets text

In [ ]:
# What values does this column contain
print(rentals["helmets_text"].value_counts(dropna=False))

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64


In [ ]:
# Create a column "child_helmet" which is `True` if the text contains the word "child"
rentals["child_helmet"] = rentals["helmets_text"].str.contains("child")

# Using RegEx to extract the number from the text if it contains it
number = rentals["helmets_text"].str.extract(r"(\d+)", expand=False).astype(float)
# Create a new column that contains the number of helmets rented with the bikes
# Fill it with the number extracted in previous step or 0 if it is "No helmets"
rentals["helmets_count"] = number.where(~rentals["helmets_text"].str.lower().str.contains("no helmets"), 0)

# Count child helmets
print("Total child helmets:", rentals["child_helmet"].sum())
# Check the values in "helmets_count"
display(rentals["helmets_count"].value_counts(dropna=False).sort_index())

Total child helmets: 47


helmets_count
0.0     59
1.0    193
2.0    127
3.0     20
NaN      4
Name: count, dtype: int64

"No helmets" has no number, so the extraction gives `NaN` for those rows. `where()` keeps the number where the text is not "no helmets" and puts `0` everywhere else, so "No helmets" becomes `0`.

The 4 rows with a missing "helmets_text" stay missing, because we do not know what they rented.

#### Task 8: Spellings and duplicates

In [ ]:
# Count how many different values there are for these columns, 
# and how many if the text is normalized
for c in ["station", "bike_type", "status"]:
    print(c, rentals[c].nunique(), rentals[c].str.strip().str.lower().nunique())

station 20 5
bike_type 3 3
status 3 3


Column "station" has the problem: 20 different values that are really 5 stations, written with extra spaces and different capital letters. Columns "bike_type" and "status" are already fine -- each has 3 different values before and after the normalization.

In [ ]:
# Fixing the problematic "station" column
rentals["station"] = rentals["station"].str.strip().str.title()
# Counting rentals per station
print(rentals["station"].value_counts())

station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64


In [ ]:
# Counting complete duplicate rows
print(rentals.duplicated().sum())
# Counting duplicate "rental_id"s
print(rentals.duplicated(subset="rental_id").sum())

3
3


In [ ]:
# Removing the duplicates and printing the final shape
rentals = rentals.drop_duplicates().reset_index(drop=True)
print(rentals.shape)

(400, 16)


There were 3 duplicate rows and 3 duplicate "rental_id"s, so the 3 duplicates are exact copies. After removing them, the shape of the dataframe is: (400, 16).